In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json, glob, pandas as pd, numpy as np, os
from pathlib import Path
from datetime import datetime

# ROOT dinámico — igual que en notebooks de insights
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(f"No se encontró requirements.txt subiendo desde {Path.cwd()}")
os.chdir(ROOT)
print(f"✅ ROOT: {ROOT}")

def ultimo_json(carpeta_base: str) -> str:
    archivos = glob.glob(f'{carpeta_base}/**/*.json', recursive=True)
    assert len(archivos) > 0, f"❌ No hay JSONs en {carpeta_base}"
    def parse_fecha(ruta):
        try:
            return datetime.strptime(Path(ruta).stem, "%d_%m_%Y")
        except ValueError:
            return datetime.min
    return max(archivos, key=parse_fecha)

# ── ambiental — rutas desde raíz del repo ──
f = ultimo_json('data/ambiental')       # ← sin ../../
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')

# Validar estructura
assert 'observacion_meteorologica' in data, \
    f"❌ Key 'observacion_meteorologica' no encontrada. Keys: {list(data.keys())}"
assert 'estaciones' in data['observacion_meteorologica'], \
    (f"❌ Key 'estaciones' no encontrada.\n"
     f"Keys en 'observacion_meteorologica': {list(data['observacion_meteorologica'].keys())}\n"
     f"La API AEMET probablemente devolvió error en la última captura.")
assert 'calidad_aire' in data, \
    f"❌ Key 'calidad_aire' no encontrada. Keys: {list(data.keys())}"

print(f"✅ Estructura JSON validada")
print(f"   Estaciones AEMET: {len(data['observacion_meteorologica']['estaciones'])}")
print(f"   Ciudades WAQI:    {len(data['calidad_aire'].get('ciudades', []))}")

✅ ROOT: /workspaces/TFG_Spain-s_Migratory_Flow
Archivo cargado: data/ambiental/2026/06/20_06_2026.json
Keys disponibles: ['timestamp', 'observacion_meteorologica', 'calidad_aire']
✅ Estructura JSON validada
   Estaciones AEMET: 9318
   Ciudades WAQI:    30


In [2]:
#============================================
# Celda 2 — Explorar estructura AEMET
#============================================
aemet_raw  = data['observacion_meteorologica']
estaciones = aemet_raw['estaciones']

print(f'Total estaciones: {aemet_raw["total_estaciones"]}')
print(f'Timestamp captura: {aemet_raw["timestamp_captura"]}')
print(f'\nKeys de una estación:')
print(list(estaciones[0].keys()))
print(f'\nEjemplo estación[0]:')
print(json.dumps(estaciones[0], indent=2, ensure_ascii=False))


Total estaciones: 9318
Timestamp captura: 2026-06-20T10:12:33.433258

Keys de una estación:
['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr']

Ejemplo estación[0]:
{
  "idema": "0002I",
  "ubi": "VANDELLÓS",
  "lat": 40.95806,
  "lon": 0.871385,
  "alt": 32.0,
  "fint": "2026-06-19T22:00:00+0000",
  "ta": 24.1,
  "tamin": 24.0,
  "tamax": 24.2,
  "prec": 0.0,
  "hr": 73.0,
  "vv": 1.9,
  "vmax": 5.2,
  "dv": 331.0,
  "dmax": 316.0,
  "pres": 1014.1,
  "pres_nmar": 1017.8,
  "vis": null,
  "inso": null,
  "ts": null,
  "tpr": 19.0
}


In [3]:
#============================================
# Celda 3 — Parsear AEMET a DataFrame
#============================================
df_aemet = pd.DataFrame(estaciones)

# Convertir timestamp
df_aemet['fint'] = pd.to_datetime(df_aemet['fint'], utc=True, errors='coerce')
df_aemet['fecha'] = df_aemet['fint'].dt.date

# Columnas numéricas
cols_num = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
            'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
for c in cols_num:
    if c in df_aemet.columns:
        df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

print(f'Shape: {df_aemet.shape}')
print(f'Columnas: {list(df_aemet.columns)}')
df_aemet.head()


Shape: (9318, 22)
Columnas: ['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr', 'fecha']


,idema,ubi,lat,lon,alt,fint,ta,tamin,tamax,prec,...,vmax,dv,dmax,pres,pres_nmar,vis,inso,ts,tpr,fecha
0,0002I,VANDELLÓS,40.958060,0.871385,32.0,2026-06-19 22:00:00+00:00,24.1,24.0,24.2,0.0,...,5.2,331.0,316.0,1014.1,1017.8,NaN,NaN,NaN,19.0,2026-06-19
1,0009X,ALFORJA,41.213892,0.963335,406.0,2026-06-19 22:00:00+00:00,18.4,18.4,19.5,0.0,...,0.9,194.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-19
2,0016A,REUS AEROPUERTO,41.145000,1.163611,71.0,2026-06-19 22:00:00+00:00,23.6,23.6,24.4,0.0,...,3.6,80.0,70.0,1009.4,1018.2,30.0,0.0,21.8,19.3,2026-06-19
3,0034X,VALLS,41.293053,1.260838,233.0,2026-06-19 22:00:00+00:00,21.9,21.9,23.2,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-19
4,0042Y,TARRAGONA FAC. GEOGRAFIA,41.123892,1.249167,55.0,2026-06-19 22:00:00+00:00,23.6,23.6,24.2,0.0,...,4.8,116.0,80.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-19


In [4]:
#============================================
# Celda 4 — Análisis rápido AEMET
#============================================
print('Valores nulos por columna:')
nulos = df_aemet.isnull().sum()
print(nulos[nulos > 0])

print(f'\nRango temperatura:  {df_aemet["ta"].min():.1f}°C — {df_aemet["ta"].max():.1f}°C')
print(f'Media temperatura:  {df_aemet["ta"].mean():.1f}°C')
print(f'\nEstaciones con precipitación > 0: {(df_aemet["prec"] > 0).sum()}')
print(f'Precipitación máxima: {df_aemet["prec"].max():.1f} mm')
print(f'\nRango altitud: {df_aemet["alt"].min():.0f}m — {df_aemet["alt"].max():.0f}m')
print(f'\nEstaciones sin coordenadas: {df_aemet[["lat","lon"]].isnull().any(axis=1).sum()}')


Valores nulos por columna:
ta            107
tamin         107
tamax         107
prec          233
hr            104
vv           1119
vmax         1123
dv           1118
dmax         1123
pres         6145
pres_nmar    7341
vis          8189
inso         7756
ts           7794
tpr          6472
dtype: int64

Rango temperatura:  8.4°C — 33.2°C
Media temperatura:  21.0°C

Estaciones con precipitación > 0: 159
Precipitación máxima: 17.4 mm

Rango altitud: 0m — 2410m

Estaciones sin coordenadas: 0


In [5]:
#============================================
# Celda 5 — Explorar estructura WAQI
#============================================
waqi_raw = data['calidad_aire']
ciudades = waqi_raw['ciudades']

print(f'Total ciudades OK:  {waqi_raw["total_ciudades_ok"]}')
print(f'Total errores:      {waqi_raw["total_errores"]}')
print(f'\nKeys de una ciudad:')
print(list(ciudades[0].keys()))
print(f'\nEjemplo ciudad[0]:')
print(json.dumps(ciudades[0], indent=2, ensure_ascii=False))


Total ciudades OK:  30
Total errores:      3

Keys de una ciudad:
['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']

Ejemplo ciudad[0]:
{
  "ciudad": "madrid",
  "nombre": "Madrid",
  "lat": 40.4167754,
  "lon": -3.7037902,
  "timestamp": "2026-06-20 07:00:00",
  "aqi": 55,
  "dominante": "pm25",
  "NO2": 4.6,
  "PM25": 55,
  "PM10": 33,
  "O3": 28.1,
  "SO2": 1.6,
  "CO": 0.1,
  "temperatura": 28.1,
  "humedad": 37.1,
  "presion": 1019.3,
  "viento": 2
}


In [6]:
#============================================
# Celda 6 — Parsear WAQI a DataFrame
#============================================
df_waqi = pd.DataFrame(ciudades)

# Convertir tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat','lon','aqi','NO2','PM25','PM10','O3','SO2','CO',
            'temperatura','humedad','presion','viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

print(f'Shape: {df_waqi.shape}')
print(f'Columnas: {list(df_waqi.columns)}')
df_waqi.head(10)


Shape: (30, 17)
Columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento
0,madrid,Madrid,40.416775,-3.703790,2026-06-20 07:00:00,55,pm25,4.6,55.0,33.0,28.1,1.6,0.1,28.1,37.1,1019.3,2.0
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-06-20 11:00:00,31,o3,8.7,34.0,19.0,30.5,1.1,0.1,28.1,48.0,1022.8,1.0
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-06-20 07:00:00,57,pm25,3.7,57.0,26.0,13.9,0.6,NaN,27.6,58.5,NaN,0.2
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-06-20 11:00:00,67,pm25,1.4,67.0,39.0,29.6,1.6,0.1,30.0,55.0,1013.7,2.3
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-06-20 09:00:00,0,co,4.6,65.0,23.0,NaN,2.6,0.1,27.0,65.0,1018.0,3.0
5,zaragoza,"El Picarral, Zaragoza, Spain",41.670278,-0.871111,2026-06-20 10:00:00,58,pm25,5.2,58.0,NaN,35.2,NaN,1.3,28.1,32.0,1020.8,3.6
6,malaga,"Carranque, Malaga, Spain",36.719640,-4.447500,2026-06-20 11:00:00,38,pm25,5.5,38.0,28.0,29.6,2.1,0.1,28.3,65.0,1016.7,7.0
7,murcia,"San Basilio Murcia Ciudad, Murcia, Spain",37.993960,-1.144628,2026-06-20 11:00:00,63,pm25,8.8,63.0,46.0,29.1,0.8,0.1,30.5,49.5,1017.3,2.0
8,palma,"foners, Baleares, Spain",39.571250,2.657028,2026-06-20 07:00:00,46,pm25,3.2,46.0,19.0,21.6,1.5,0.1,33.0,32.5,1019.0,6.1
9,alicante,"Rabassa, Alacant, Valencia, Spain",38.351111,-0.513889,2026-06-20 07:00:00,22,o3,3.7,13.0,9.0,22.4,0.6,0.1,28.5,61.5,1017.0,3.3


In [7]:
#============================================
# Celda 7 — Análisis rápido WAQI
#============================================
def nivel_aqi(aqi):
    if pd.isna(aqi):       return 'Sin datos'
    elif aqi <= 50:        return '🟢 Bueno'
    elif aqi <= 100:       return '🟡 Moderado'
    elif aqi <= 150:       return '🟠 Insalubre sensibles'
    elif aqi <= 200:       return '🔴 Insalubre'
    else:                  return '🟣 Muy insalubre'

df_waqi['nivel_aqi'] = df_waqi['aqi'].apply(nivel_aqi)

print('Ciudades por nivel AQI:')
print(df_waqi['nivel_aqi'].value_counts().to_string())

print(f'\nTop 5 peor AQI:')
print(df_waqi.nlargest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nTop 5 mejor AQI:')
print(df_waqi.nsmallest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nValores nulos por columna:')
print(df_waqi.isnull().sum()[df_waqi.isnull().sum() > 0])


Ciudades por nivel AQI:
nivel_aqi
🟢 Bueno       22
🟡 Moderado     8

Top 5 peor AQI:
                                   nombre  aqi dominante  PM25  PM10   O3
                  Ranilla, Sevilla, Spain   67      pm25  67.0  39.0 29.6
 San Basilio Murcia Ciudad, Murcia, Spain   63      pm25  63.0  46.0 29.1
                  La Orden, Huelva, Spain   63      pm25  63.0  35.0 15.9
             El Picarral, Zaragoza, Spain   58      pm25  58.0   NaN 35.2
Pista de Silla, València, Valencia, Spain   57      pm25  57.0  26.0 13.9

Top 5 mejor AQI:
                                     nombre  aqi dominante  PM25  PM10    O3
       Mazarredo, Bilbao, País Vasco, Spain    0        co  65.0  23.0   NaN
  Vila Velha - IBES, Espírito Santo, Brazil    5        o3   NaN   NaN  4.73
Girona (Escola de Música), Catalunya, Spain   15      pm10  25.0  15.0 16.00
         Lope de Vega, Vigo, Galicia, Spain   16      pm25  16.0  14.0  5.70
    Tarragona (Bonavista), Catalunya, Spain   17      pm10  30.0  17

In [8]:
#============================================
# Celda 8 — Resumen calidad de datos
#============================================
resumen = pd.DataFrame([
    {
        'tabla':     'aemet_estaciones',
        'filas':     len(df_aemet),
        'columnas':  len(df_aemet.columns),
        'periodos':  df_aemet['fint'].nunique(),
        'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
    },
    {
        'tabla':     'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique(),
        'nulos_%':   round(df_waqi.isnull().sum().sum()  / (df_waqi.shape[0]  * df_waqi.shape[1])  * 100, 1)
    },
])
print(resumen.to_string(index=False))


           tabla  filas  columnas  periodos  nulos_%
aemet_estaciones   9318        22        12     23.8
   waqi_ciudades     30        18         9      3.7


In [9]:
#============================================
# Celda 9 — Guardar parquets
#============================================
os.makedirs('output/ambiental/01-raw', exist_ok=True)   # ← sin ../../

df_aemet.to_parquet('output/ambiental/01-raw/raw_aemet_estaciones.parquet', index=False)
print(f'✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet ({len(df_aemet)} filas)')

df_waqi.to_parquet('output/ambiental/raw_waqi_ciudades.parquet', index=False)
print(f'✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet ({len(df_waqi)} filas)')

print('\n🎉 Todos los parquets ambiental guardados en output/')

✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet (9318 filas)
✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet (30 filas)

🎉 Todos los parquets ambiental guardados en output/
